# Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
import plotly.graph_objs as go
import plotly.express as px
import cufflinks as cf
from plotly.subplots import make_subplots

init_notebook_mode(connected=True)  # For Notebooks

cf.go_offline() # For offline use

# Loading and Inspecting Dataset

In [ ]:
df = pd.read_csv('/Users/festusattornelson/Documents/Projects/Python MSIT/car_data.csv')
df.head(2)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Checkimg and dropping duplicate rows
df.duplicated().sum()

In [ ]:
#Checking for missing values
df.isna().sum()

In [ ]:
def initial_clean_data(data):
    """
      Cleans a given DataFrame by:
    - Removing duplicate rows
    - Standardizing text columns (trimming spaces, converting to lowercase)
    - datatype conversion (Year to datetime)
    - Drop missing values in Market Category
    - Filter data by year >= 1995

    Parameters:
    df (pd.DataFrame): The input DataFrame to be cleaned

    Returns:
    pd.DataFrame: The cleaned DataFrame

    """
    df = data.copy()

    #Remove duplicates
    df = df.drop_duplicates(keep='first').reset_index(drop=True)

    #Filter columns based on year >= 1995
    df = df.loc[df.Year >= 1995].reset_index(drop=True)

    #Drop missing values in Market Category
    #df['Market Category'] = df['Market Category'].replace("", np.nan)
    #df = df.dropna(subset=['Market Category']).reset_index(drop=True)

    #Convert year to datetime
    df['Year'] = pd.to_datetime(df['Year'], format='%Y')

    #Format text columns
    df.columns = [col.replace(' ', '').replace('_', '') for col in df.columns]

    df = df.map(lambda row: row.lower() if isinstance(row, str) else row)


    return df

In [ ]:
# Clean the data
df_car = initial_clean_data(data=df)

print("Initial clean dataframe:")
#df_car.head()

In [ ]:
df_car.shape

In [ ]:
df_car.info()

***categorical and numerical columns***

In [ ]:
categorical_columns = []
numerical_columns = []

for col in df_car.columns:
  if df_car[col].dtype == 'object':
    categorical_columns.append(col)
  else:
    numerical_columns.append(col)

In [ ]:
categorical_columns

In [ ]:
numerical_columns

# Data Cleansing

**Handing Missing values**

In [ ]:
# Checking for missing values
missing_values = df_car.isna().sum()
percent_missing_values = (df_car.isna().sum()/df_car.shape[0])*100
missing_values_summary= pd.DataFrame({'missing values': missing_values, 'missing_values (%)': percent_missing_values})
missing_values_summary

#### Column: EngineCylinders

In [ ]:
df_car.loc[(df_car.EngineCylinders.isna())].head()

In [ ]:
df_car.columns

In [ ]:
#Electric cars don't have engine cylinders = 0
df_car.loc[(df_car["TransmissionType"].fillna("") == "direct_drive") &(df_car["EngineFuelType"].fillna("") == "electric"),
    "EngineCylinders"] = 0

In [ ]:
#Mazda RX-7or8 has rotary (wankel) engine hence 0 conventional cylinders
df_car.loc[df_car["Model"].isin(["rx-7", "rx-8"]), "EngineCylinders"] = 0

In [ ]:
df_car.isna().sum()

#### Column: EngineHP

In [ ]:
def fill_engine_hp(df):
    # Fill by similar cars (Make and Model)
    df["EngineHP"] = df.groupby(["Make", "Model"])["EngineHP"].transform(lambda x: x.fillna(x.median()))

    # Fill remaining with median of the make
    df["EngineHP"] = df.groupby("Make")["EngineHP"].transform(lambda x: x.fillna(x.median()))

    # Fill remaining with median

    df_car['EngineHP'] = df_car['EngineHP'].fillna(df_car['EngineHP'].median())

    return df

In [ ]:
# Fill EngineHP
df_car = fill_engine_hp(df_car)

print("Imputing EngineHP:")
df_car.isna().sum()#.sum()

#### Column: NumberofDoors

In [ ]:
df_car.loc[(df_car.NumberofDoors.isna())]

In [ ]:
#Fill by mode of the model of the car
df_car["NumberofDoors"] = df_car.groupby("Model")["NumberofDoors"].transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else np.nan))

In [ ]:
df_car.isna().sum()

#### Column: MarketCategory

In [ ]:
df_car.MarketCategory.value_counts()

- percent missing values of Market category is 39% which will significantly reduce the quantity of dataset
- replace missing values of market category based on make or mode and else replace with unknown

In [ ]:
#Fill the missing values of marketcategory based on mode or make
df_car['MarketCategory'] = df_car.groupby(['Make', 'Model'])['MarketCategory'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 'Unknown')
)

In [ ]:
df_car.isna().sum()

#### Column: EngineFuelType

In [ ]:
df_car['EngineFuelType'] = df_car.groupby('Make')['EngineFuelType'].transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else None))

In [ ]:
df_car.isna().sum()

# 2. Feature Engineering

#### Creating new columns: Price per HP (MSRP / Engine HP)

In [ ]:
# No zero divisible error
# df_car['PriceperHP'] = df_car['MSRP'] / df_car['EngineHP']

In [ ]:
# safe handling of potential division by zero error 
df_car['Price_per_HP'] = df_car.apply(
    lambda row: round(row['MSRP'] / row['EngineHP'],2) if row['EngineHP'] > 0 else float('nan'), axis=1)

# summary statistics for the new column: 'PriceperHP'
print("\nMSRP per HP Statistics:")
print(df_car['Price_per_HP'].describe())

print("\nSample of cars with their MSRP per HP:")
print(df_car[['Make', 'Model', 'MSRP', 'EngineHP', 'Price_per_HP']].sample(5))

In [ ]:
df_car.isna().sum()

#### Creating new columns: Total MPG (average of city mpg and highway mpg)

In [ ]:
# Create the new column for total MPG (average of city and highway)
df_car['total_MPG'] = round((df_car['citympg'] + df_car['highwayMPG']) / 2, 2)

# summary statistics for the new column: 'totalMPG'
print("\nTotal MPG Statistics:")
print(df_car['total_MPG'].describe())

print("\nSample of cars with their MPG values:")
print(df_car[['Make', 'Model', 'citympg', 'highwayMPG', 'total_MPG']].sample(5))

print("\nTop 5 cars by total MPG:")
top_mpg = df_car.sort_values('total_MPG', ascending=False) # cars with the best total MPG
print(top_mpg[['Make', 'Model', 'citympg', 'highwayMPG', 'total_MPG']].head(5))

In [ ]:
df_car.isna().sum()

#### Creating new columns: Car Age 

In [ ]:
def new_features(data):
  """
    creates 2 new columns of the DataFrame by:
    - avg_mpg (average of city and highway)
    - price_per_hp (MSRP divided by EngineHP
    - car_age (current year - year)

    Parameters to create new columns:
    df_car.citympg, df_car.highwayMPG, df_car.MSRP and df_car.EngineHP.

    Returns:
    pd.DataFrame: with 2 new columns

    """
  # Create the new column for total MPG (average of city and highway)
  df_car['total_MPG'] = round((df_car['citympg'] + df_car['highwayMPG']) / 2, 2)

  # Dealing of potential zerodivisionerror
  df_car['Price_per_HP'] = df_car.apply(
    lambda row: round(row['MSRP'] / row['EngineHP'],2) if row['EngineHP'] > 0 else float('nan'), axis=1)

  #Depreciation trends
  current_year = 2026

  df_car["Car_Age"] = current_year - df_car["Year"].dt.year

  #Rename some columns
  #df_car.columns = [col.title().replace("Hp", "HP").replace("Mpg", "MPG").replace("Msrp", "MSRP") for col in df_car.columns]

  return df_car

In [ ]:
# new columns
df_car = new_features(data=df_car)

print("New columns in dataframe:")
df_car.head(5)

# 3. Exploratory Data Analysis (EDA)

#### **Descriptive Analysis**

In [ ]:
df_car.columns

In [ ]:
columns_summary = ['EngineHP', 'MSRP', 'Popularity', 'highwayMPG', 'citympg']

In [ ]:
summary_stats = df_car[columns_summary].describe()
summary_stats

In [ ]:
focused_stats = summary_stats.loc[['mean', '50%', 'std']]
focused_stats = focused_stats.rename(index={'50%': 'median'})
focused_stats.T #transpose the results

#### **Grouped Analysis**

****Checking mean, std and median for some individual parameters****

- VehicleSize
- DrivenWheels
- EngineCylinders

In [ ]:
df_VehicleSize = df_car.groupby(['VehicleSize'])[['MSRP', 'Popularity']].agg(['mean', 'median', 'std'])
df_VehicleSize.columns = ['_'.join(col) for col in df_VehicleSize.columns]
df_VehicleSize = df_VehicleSize.reset_index()
df_VehicleSize

In [ ]:
df_DrivenWheels = df_car.groupby(['DrivenWheels'])[['MSRP', 'Popularity']].agg(['mean', 'median', 'std'])
df_DrivenWheels.columns = ['_'.join(col) for col in df_DrivenWheels.columns]
df_DrivenWheels = df_DrivenWheels.reset_index()
df_DrivenWheels

In [ ]:
df_EngineCylinders = df_car.groupby(['EngineCylinders'])[['MSRP', 'Popularity']].agg(['mean', 'median', 'std'])
df_EngineCylinders.columns = ['_'.join(col) for col in df_EngineCylinders.columns]
df_EngineCylinders = df_EngineCylinders.reset_index()
df_EngineCylinders

****Checking mean, std and median by grouping them****
- VehicleSize
- EngineCylinders
- DrivenWheels

In [ ]:
grouped_stats = df_car.groupby(['DrivenWheels', 'VehicleSize', 'EngineCylinders'])[['MSRP', 'Popularity']].mean()#.reset_index()
grouped_stats.head(10)

In [ ]:
cars_per_group = df_car.groupby(['DrivenWheels', 'VehicleSize', 'EngineCylinders']).size().reset_index(name='car_count')
print("\nNumber of cars in each group:")
print(cars_per_group.head(10))

In [ ]:
# Select the row(s) with maximum car_count
max_count_row = cars_per_group[cars_per_group['car_count'] == cars_per_group['car_count'].max()]

print("\nGroup(s) with the maximum number of cars:")
print(max_count_row)

***Sort by average MSRP to see which combinations are most expensive***

In [ ]:
print("\nGroups sorted by average MSRP (descending):")
print(grouped_stats.sort_values('MSRP', ascending=False).head(10))

***Sort by average Popularity to see which combinations are most popular***

In [ ]:
print("\nGroups sorted by average Popularity (descending):")
print(grouped_stats.sort_values('Popularity', ascending=False).head(10))

***Type of Engine Fuel on the Highway***

In [ ]:
df_car.groupby('EngineFuelType')['highwayMPG'].mean().sort_values(ascending=False)

***Type of Engine Fuel in the city***

In [ ]:
df_car.groupby('EngineFuelType')['citympg'].mean().sort_values(ascending=False)

In [ ]:
df_car.groupby(["TransmissionType"])["EngineHP"].median()

#### **Visualizations**

##### Histogram

In [ ]:
#raw vs transformed data
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

sns.histplot(df_car["MSRP"], bins=80, kde=True, ax=ax[0])
ax[0].set_title("Distribution of MSRP")

sns.histplot(np.log(df_car["MSRP"]), bins=80, kde=True)
ax[1].set_title("Distribution of Log MSRP")

plt.show()

In [ ]:
#raw vs transformed data
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

sns.histplot(df_car["citympg"], bins=80, kde=True, ax=ax[0])
ax[0].set_title("Distribution of Citympg")

sns.histplot(df_car["highwayMPG"], bins=80, kde=True)
ax[1].set_title("Distribution of Highway")

plt.show()

In [ ]:
#raw vs transformed data
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

sns.histplot(df_car["MSRP"], bins=80, kde=True, ax=ax[0])
ax[0].set_title("Distribution of MSRP")

sns.histplot(np.log(df_car["MSRP"]), bins=80, kde=True)
ax[1].set_title("Distribution of Log MSRP")

plt.show()

In [ ]:
# Melt the dataframe to long format for easy faceting
df_mpg = df_car.melt(
    value_vars=['citympg', 'highwayMPG'],
    var_name='MPG_Type',
    value_name='MPG'
)

# Interactive histogram with facets
fig = px.histogram(
    df_mpg,
    x='MPG',
    facet_col='MPG_Type',              # facet by city vs highway
    color_discrete_sequence=['#636EFA'],
    nbins=40,
    title='Distribution of City and Highway MPG',
    labels={'MPG':'Miles per Gallon'}
)

fig.update_layout(showlegend=False)
fig.show()

##### A bar chart showing the average MSRP for each category in Vehicle Size

In [ ]:
# Compute average MSRP per Vehicle Size
avg_msrp = df_car.groupby('VehicleSize')['MSRP'].mean().reset_index()

# Set style
sns.set_style("whitegrid")

# Plot bar chart
plt.figure(figsize=(8,5))
sns.barplot(data=avg_msrp, hue='VehicleSize', y='MSRP', palette='coolwarm')

# Labels and title
plt.xlabel("Vehicle Size")
plt.ylabel("Average MSRP")
plt.title("Average MSRP by Vehicle Size")
plt.show()

##### A scatter plot showing the relationship between Engine HP and MSRP.

In [ ]:
# Create a scatter plot showing the relationship between Engine HP and MSRP
fig = px.scatter(
    df_car,  # Using the original dataframe
    x='EngineHP',
    y='MSRP',
    title='Relationship between Engine Horsepower and MSRP',
    labels={'EngineHP': 'Engine Horsepower', 'MSRP': 'Manufacturer Suggested Retail Price ($)'},
    opacity=0.7,
    color_discrete_sequence=['#636EFA']
)

fig.update_layout(
    xaxis_title='Engine Horsepower (HP)',
    yaxis_title='MSRP ($)'
)

fig.show()

In [ ]:
# Create a scatter plot showing the relationship between Engine HP and MSRP using Seaborn

# Set the style
sns.set_style("whitegrid")

# Create the figure and axes
plt.figure(figsize=(10, 6))

# Create the scatter plot
sns.scatterplot(data=df_car, x='EngineHP', y='MSRP', alpha=0.7, color='#636EFA')

# Set the title and labels
plt.title('Relationship between Engine Horsepower and MSRP', fontsize=14)
plt.xlabel('Engine Horsepower (HP)', fontsize=12)
plt.ylabel('Manufacturer Suggested Retail Price ($)', fontsize=12)

# Improve the layout
plt.tight_layout()

# Show the plot
plt.show()

##### A boxplot showing the distribution of MSRP for each category in Driven_Wheels.

In [ ]:
# Set the style
sns.set_style("whitegrid")

# Create the figure with a larger size to accommodate all categories
plt.figure(figsize=(12, 8))

sns.boxplot(data=df_car, hue='DrivenWheels', y='MSRP', palette='coolwarm')
plt.title('Distribution of MSRP by Driven Wheels Type', fontsize=14)
plt.xlabel('Driven Wheels', fontsize=12)
plt.ylabel('Manufacturer Suggested Retail Price ($)', fontsize=12)

plt.xticks(rotation=45)  # Rotate x-axis labels 
plt.tight_layout()   # Improve the layout
plt.show()

In [ ]:
# Create the boxplot
fig = px.box(df_car, x='DrivenWheels', y='MSRP',
    title='Distribution of MSRP by Driven Wheels Type',
    labels={
        'DrivenWheels': 'Driven Wheels', 'MSRP': 'Manufacturer Suggested Retail Price ($)'}, color='DrivenWheels',  
    category_orders={'DrivenWheels': sorted(df_car['DrivenWheels'].unique())})

# Update layout for better appearance
fig.update_layout(xaxis_title='Driven Wheels', yaxis_title='MSRP ($)',showlegend=False, height=600, width=900)

# Show the plot
fig.show()

##### Correlation Ananalysis

In [ ]:
df_car['EngineHP'] = pd.to_numeric(df_car['EngineHP'], errors='coerce')

In [ ]:
#df_car.info()

In [ ]:
variables = ['EngineHP', 'MSRP', 'Popularity', 'citympg', 'highwayMPG']
df_corr = df_car[variables].copy()

# Calculate the correlation matrix
corr_matrix = df_corr.corr()

corr_matrix

In [ ]:
# Create a heatmap to visualize correlations
fig = px.imshow(
    corr_matrix,
    text_auto=True,  # Show correlation values
    color_continuous_scale='RdBu_r',  # Red-Blue scale (negative to positive)
    zmin=-1,  # Minimum correlation value
    zmax=1,   # Maximum correlation value
    title='Correlation Matrix: Engine HP, MSRP, Popularity, city mpg, and highway MPG')

# Improve layout
fig.update_layout(
    width=800,
    height=700,
    xaxis_title='',
    yaxis_title='')

# Show the heatmap
fig.show()

In [ ]:
# Investigate correlation between Engine HP, MSRP, Popularity, city mpg, and highway MPG using Seaborn

# Set the style
sns.set(style="white")

# Select the variables of interest
variables = ['EngineHP', 'MSRP', 'Popularity', 'citympg', 'highwayMPG']
df_corr = df_car[variables].copy()

plt.figure(figsize=(10, 8))

corr_matrix = df_corr.corr()

# Create a mask for the upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# Create the heatmap
heatmap = sns.heatmap(
    corr_matrix, 
    annot=True,           # Show correlation values
    fmt=".2f",            # Format as 2 decimal places
    cmap='RdBu_r',        # Red-Blue scale (negative to positive)
    vmin=-1,              # Minimum correlation value
    vmax=1,               # Maximum correlation value
    mask=mask,            # Apply the mask to show only lower triangle
    linewidths=0.5,       # Add lines between cells
    cbar_kws={"shrink": .8}
)

# Set title
plt.title('Correlation Matrix: Engine HP, MSRP, Popularity, city mpg, and highway MPG', fontsize=14)
plt.tight_layout()

# Show the heatmap
plt.show()

##### Finding outliers of MSRP

In [ ]:
plt.figure(figsize=(10,6))

# Reshape data
df_box = df_car[["MSRP"]].melt(var_name="Feature",value_name="Value")

sns.boxplot(data=df_box, hue="Feature", y="Value", palette="Set2")

plt.title("Boxplot of MSRP", fontsize=16)
plt.xlabel("Feature", fontsize=14)
plt.ylabel("Value", fontsize=14)

plt.show()

In [ ]:
plt.figure(figsize=(12,6))

# Calculate Q1, Q3, and IQR for MSRP
Q1 = df_car['MSRP'].quantile(0.25)
Q3 = df_car['MSRP'].quantile(0.75)
IQR = Q3 - Q1

# Define outliers thresholds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Identify outliers
outliers = df_car[(df_car['MSRP'] < lower_bound) | (df_car['MSRP'] > upper_bound)]

sns.scatterplot(data=df_car, x="EngineHP", y="MSRP", color="blue", alpha=0.5)

sns.scatterplot(data=outliers, x="EngineHP", y="MSRP", color="red", label="Outliers")

plt.title("Outliers in MSRP (Highlighted in Red)")
plt.xlabel("Engine HP")
plt.ylabel("MSRP")

plt.legend()
plt.show()

##### Barplot (transmission vs popularity, vehicle size vs popularity)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 6))

# Calculate the order for TransmissionType based on Popularity
transmission_order = df_car.groupby('TransmissionType')['Popularity'].mean().sort_values(ascending=False).index

# Plot 1: TransmissionType vs Popularity
sns.barplot(data=df_car, x="TransmissionType", y="Popularity", palette="colorblind", errorbar=None, ax=ax[0],order=transmission_order)

ax[0].set_title("Transmission Type vs Popularity", fontsize=16)
ax[0].set_xlabel("Transmission Type", fontsize=14)
ax[0].set_ylabel("Popularity", fontsize=14)
ax[0].tick_params(axis='x', rotation=45)  # rotate x labels

# Calculate the order for VehicleSize based on Popularity
vehicle_size_order = df_car.groupby('VehicleSize')['Popularity'].mean().sort_values(ascending=False).index

# Plot 2: VehicleSize vs Popularity
sns.barplot(data=df_car, x="VehicleSize", y="Popularity", palette="colorblind", errorbar=None, ax=ax[1], order=vehicle_size_order)

ax[1].set_title("Vehicle Size vs Popularity", fontsize=16)
ax[1].set_xlabel("Vehicle Size", fontsize=14)
ax[1].set_ylabel("Popularity", fontsize=14)
ax[1].tick_params(axis='x', rotation=45)  # rotate x labels

plt.tight_layout()
plt.show()

In [ ]:
fig = make_subplots(rows=1, cols=2, 
                    subplot_titles=("Transmission Type vs Popularity", 
                                    "Vehicle Size vs Popularity"))

# Calculate average popularity by transmission type and sort
transmission_popularity = df_car.groupby('TransmissionType')['Popularity'].mean().reset_index()
transmission_popularity = transmission_popularity.sort_values('Popularity', ascending=False)

# Calculate average popularity by vehicle size and sort
vehicle_size_popularity = df_car.groupby('VehicleSize')['Popularity'].mean().reset_index()
vehicle_size_popularity = vehicle_size_popularity.sort_values('Popularity', ascending=False)

# Add bar chart for Transmission Type
fig.add_trace(
    go.Bar(
        x=transmission_popularity['TransmissionType'],
        y=transmission_popularity['Popularity'],
        marker_color=px.colors.qualitative.Bold,
        hovertemplate='<b>Transmission Type:</b> %{x}<br>' +
                      '<b>Popularity:</b> %{y:.1f}<extra></extra>',
        name='Transmission Type'),row=1, col=1)

# Add bar chart for Vehicle Size
fig.add_trace(
    go.Bar(
        x=vehicle_size_popularity['VehicleSize'],
        y=vehicle_size_popularity['Popularity'],
        marker_color=px.colors.qualitative.Pastel,
        hovertemplate='<b>Vehicle Size:</b> %{x}<br>' +
                      '<b>Popularity:</b> %{y:.1f}<extra></extra>',
        name='Vehicle Size'),row=1, col=2)

# Update layout
fig.update_layout(
    title_text="Car Popularity Analysis", height=600, width=1000, showlegend=False,
    hoverlabel=dict(bgcolor="white",  font_size=12, font_family="Arial"))

# Update x and y axis properties
fig.update_xaxes(title_text="Transmission Type", row=1, col=1, tickangle=45)
fig.update_xaxes(title_text="Vehicle Size", row=1, col=2, tickangle=45)
fig.update_yaxes(title_text="Average Popularity", row=1, col=1)
fig.update_yaxes(title_text="Average Popularity", row=1, col=2)

# Show the interactive plot
fig.show()

##### Time Series Analysis

***Popularity by year and vehicle size***

In [ ]:
df_trend = df_car.groupby(["Year", "VehicleSize"])["Popularity"].mean().reset_index()

plt.figure(figsize=(12,6))
sns.lineplot(data=df_trend, x="Year", y="Popularity", hue="VehicleSize", marker="o")

plt.title("Popularity Trend by Vehicle Size Over Years", fontsize=16)
plt.xlabel("Year", fontsize=14)
plt.ylabel("Average Popularity", fontsize=14)
plt.legend(title="Vehicle Size")
plt.grid(True)
plt.show()

***Retail trend ny MSRP Over the years*** 

In [ ]:
df_trend = df_car.groupby(["Year", "VehicleSize"])["MSRP"].mean().reset_index()

plt.figure(figsize=(12,6))
sns.lineplot(data=df_trend, x="Year", y="MSRP", hue="VehicleSize", marker="o")

plt.title("Retail Price Trend by Vehicle Size Over Years", fontsize=16)
plt.xlabel("Year", fontsize=14)
plt.ylabel("Average MSRP", fontsize=14)
plt.legend(title="Vehicle Size")
plt.grid(True)
plt.show()

****mean MSRP by vehicle size between 1995 and 2000****

In [ ]:
#Average MSRP by year between 1995-2000
mean_msrp_year = (df_car[df_car["Year"].dt.year.between(1995, 2000)].groupby(["Year", "VehicleSize"])["MSRP"].mean())#.reset_index().sort_values(by=["Year", "VehicleSize"]))

print(mean_msrp_year)

In [ ]:
mean_msrp = (df_car[df_car["Year"].dt.year.between(1995, 2000)].groupby(["Year", "VehicleSize"])["MSRP"].mean().reset_index())
print(mean_msrp.head())

****MSRP Analysis between 1995-2000****

In [ ]:
plt.figure(figsize=(12,6))
sns.lineplot(data=mean_msrp, x="Year", y="MSRP", hue="VehicleSize", marker="o")

plt.title("Retail Price Trend by Vehicle Size (1995-2000)", fontsize=16)
plt.xlabel("Year", fontsize=14)
plt.ylabel("Average MSRP", fontsize=14)
plt.legend(title="Vehicle Size")
plt.grid(True)
plt.show()

****Depreciation of MSRP with age****

In [ ]:
#raw vs transformed data
fig, ax = plt.subplots(1, 2, figsize=(16, 5))

sns.lineplot(data=df_car, x="Car_Age", y="MSRP", hue='VehicleSize', estimator="mean", errorbar=None, ax=ax[0])
ax[0].set_xlabel("Car Age")  
ax[0].set_ylabel("MSRP")    
ax[0].set_title("Age Depreciation Trend", fontsize=16)

sns.lineplot(data=df_car, x="Car_Age", y=np.log1p(df_car["MSRP"]), hue='VehicleSize', estimator="mean", errorbar=None, ax=ax[1])
ax[1].set_title("Log-Scaled Age Depreciation Trend", fontsize=16)
ax[1].set_xlabel("Car Age")  
ax[1].set_ylabel("Log(MSRP)")  

plt.grid(True)
plt.show()

In [ ]:
mean_msrp_vs = (df_car[df_car["Year"].dt.year.between(1995, 2000)].groupby("VehicleSize")["MSRP"].mean().sort_values(ascending=False))

print(mean_msrp_vs)

##### Scatterplot

In [ ]:
corr_by_size = df_car.groupby("VehicleSize").apply(lambda x: x["EngineHP"].corr(x["MSRP"]))
corr_df = corr_by_size.reset_index()
corr_df.columns = ["VehicleSize", "Correlation"]
print(corr_df)

In [ ]:
sns.scatterplot(data=df_car, x="EngineHP", y="MSRP", hue="VehicleSize")
plt.show()

#### **Further Insights**

***Business Insights***

*****Market Segmentation*****

In [ ]:
plt.figure(figsize=(8,6))
sns.countplot(data=df_car,x="VehicleSize", palette="colorblind",
    order=df_car["VehicleSize"].value_counts().index)

plt.title("Market Segmentation by Vehicle Size", fontsize=16)
plt.xlabel("Vehicle Size", fontsize=14)
plt.ylabel("Number of Vehicles", fontsize=14)
plt.grid(axis="y")
plt.show()

*****Top and Bottom Make per Price*****

In [ ]:
# Get top 5 most expensive brands
top_avg_price_make = df_car.groupby("Make")["MSRP"].mean().sort_values(ascending=False)
top_makes = top_avg_price_make.head(5).index

In [ ]:
# Get bottom 5 least expensive brands
bottom_avg_price_make = df_car.groupby("Make")["MSRP"].mean().sort_values(ascending=False)
bottom_makes = bottom_avg_price_make.tail(5).index

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 6))

# Barplot 1: TransmissionType vs Popularity
sns.barplot(data=df_car[df_car["Make"].isin(top_makes)],x="Make",y="MSRP",
    estimator="mean",palette="colorblind",order=top_makes, errorbar=None, ax=ax[0])

ax[0].set_title("Average MSRP Make (Top 5)", fontsize=16)
ax[0].set_xlabel("Car Make", fontsize=14)
ax[0].set_ylabel("Average MSRP ($)", fontsize=14)

# Barplot 2: VehicleSize vs Popularity
sns.barplot(data=df_car[df_car["Make"].isin(bottom_makes)],x="Make",y="MSRP",
    estimator="mean",palette="colorblind",order=bottom_makes, errorbar=None, ax=ax[1])

ax[1].set_title("Average MSRP Make (Bottom 5)", fontsize=16)
ax[1].set_xlabel("Car Make", fontsize=14)
ax[1].set_ylabel("Average MSRP ($)", fontsize=14)

plt.tight_layout()  

plt.show()

***Consumer Insights***

In [ ]:
plt.figure(figsize=(12,6))
df_car.groupby("VehicleStyle")["total_MPG"].mean().sort_values(ascending=True).plot(kind="bar")
plt.xticks(rotation=45) 
plt.show()

In [ ]:
avg_mpg_make = df_car.groupby("Make")["total_MPG"].mean()

In [ ]:
avg_mpg_make.sort_values(ascending=True).plot(kind="barh",figsize=(10,8))

plt.title("Fuel Efficiency by Car Brand", fontsize=16)
plt.xlabel("Average MPG", fontsize=14)
plt.ylabel("Car Make", fontsize=14)

#plt.yticks(rotation=45)
plt.grid(False)
plt.show()

In [ ]:
plt.figure(figsize=(10,6))

sns.barplot(data=df_car, x="NumberofDoors", y="MSRP",
    hue="VehicleSize",palette="colorblind", errorbar=None)

plt.title("Practicability Vs Cost", fontsize=16)
plt.xlabel("Number of Doors", fontsize=14)
plt.ylabel("MSRP ($)", fontsize=14)

plt.show()

In [ ]:
df_car[df_car["NumberofDoors"].isin([2,3,4])].groupby("NumberofDoors")["Make"].unique()

## Encoding Categorical Columns

In [ ]:
df_car.head(2)

### Encoding: VehicleSize

In [ ]:
df_car.VehicleSize.value_counts()

In [ ]:
freq_table = df_car['VehicleSize'].value_counts()
rank_table = freq_table.rank(method='first', ascending=False).astype(int) - 1
df_car['VehicleSize_freq'] = df_car['VehicleSize'].map(rank_table)

In [ ]:
print(df_car[['VehicleSize', 'VehicleSize_freq']].drop_duplicates().sort_values('VehicleSize_freq'))

### Encoding: VehicleStyle

In [ ]:
df_car.VehicleStyle.value_counts()

In [ ]:
#freq_table = df_car['VehicleStyle'].value_counts()
#rank_table = freq_table.rank(method='first', ascending=False).astype(int) - 1
#df_car['VehicleStyle_freq'] = df_car['VehicleStyle'].map(rank_table)

In [ ]:
df_car['VehicleStyle_freq'] = df_car['VehicleStyle'].map(df_car['VehicleStyle'].value_counts().rank(method='first', ascending=False).astype(int) - 1)

In [ ]:
print(df_car[['VehicleStyle', 'VehicleStyle_freq']].drop_duplicates().sort_values('VehicleStyle_freq'))

### Encoding TransmissionType

In [ ]:
df_car.TransmissionType.value_counts()

In [ ]:
df_car['TransmissionType_freq'] = df_car['TransmissionType'].map(df_car['TransmissionType'].value_counts().rank(method='first', ascending=False).astype(int) - 1)

In [ ]:
print(df_car[['TransmissionType', 'TransmissionType_freq']].drop_duplicates().sort_values('TransmissionType_freq'))

### Encoding: DrivenWheels

In [ ]:
df_car.DrivenWheels.value_counts()

In [ ]:
df_car['DrivenWheels_freq'] = df_car['DrivenWheels'].map(df_car['DrivenWheels'].value_counts().rank(method='first', ascending=False).astype(int) - 1)

In [ ]:
print(df_car[['DrivenWheels', 'DrivenWheels_freq']].drop_duplicates().sort_values('DrivenWheels_freq'))

### Encoding: EngineFuelType

In [ ]:
df_car.EngineFuelType.value_counts()

In [ ]:
df_car['EngineFuelType_freq'] = df_car['EngineFuelType'].map(df_car['EngineFuelType'].value_counts().rank(method='first', ascending=False).astype(int) - 1)

In [ ]:
print(df_car[['EngineFuelType', 'EngineFuelType_freq']].drop_duplicates().sort_values('EngineFuelType_freq'))

In [ ]:
df_car.head(5)

In [ ]:
import pandas as pd
import numpy as np

# Create the initial dataframe with color column
color = ["red", "green", "blue", "blue", "red", "red"]
df = pd.DataFrame({"color": color})

# Calculate frequency of each color
freq_table = df['color'].value_counts()

# Map each color to its frequency count
df['color_freq'] = df['color'].map(freq_table)

# Display the result
print("Original dataframe with frequency encoding:")
print(df)

# Verify the frequencies
print("\nFrequency table:")
print(freq_table)